# Soldani - Third task - Benchmark on Thor

## Context: what this notebook adds

`2_1_benchmark_gpt.ipynb` asked the LLM to compute the causal fairness effects directly from a raw CSV dump, and `2_2_benchmark_different_efforts.ipynb` repeated that setup while sweeping the reasoning effort of `o3-mini`. Both share the same weakness: the model has to derive the conditional probabilities itself, over hundreds of (confounder, mediator) combinations, before it can even begin applying the identification formulae.

This notebook is where the corrections discussed with the supervisor land. Three things change:

- **The model no longer sees raw rows.** The prompt carries five pre-aggregated conditional probability tables - P(Y|X), P(Z), P(Y|X,Z), P(W|X,Z), P(Y|X,W,Z) - queried from the *same* fitted Bayesian Network that produces the ground truth (the supervisor's Point 4). Previously the tables came from raw `pandas` frequencies while the ground truth came from a smoothed BN, so the two sides could disagree on a sparse cell even when both applied the formula correctly.
- **Cardinality is cut before the prompt is built.** `hours-per-week` is binned into 5 ranges and `education` is grouped from 16 levels into 5 tiers, bringing the (z,w) combinations from 80 down to 25. With 80, Qwen2.5-14B did not finish DE and IE: the response was truncated before the final JSON even at `max_tokens=16384`, and compressing the output format made things worse - the model stopped computing and returned invented values.
- **SE is not asked of the model.** It is fully determined by `SE = TV - TE` (Eq. 3, Plečko and Bareinboim 2024) and is derived afterwards on both sides, so the SE row reflects only the model's TV and TE errors rather than a redundant extra computation.

The model is Qwen2.5-14B served by a local `llama.cpp` instance rather than a hosted API. Execution happens on the Thor cluster: `run_benchmark.sbatch` locates the running `llama_server_gpu` job, exports its address, and executes this notebook with `jupyter nbconvert --to notebook --execute`, writing the results to a separate `benchmark_output_<jobid>.ipynb`.

## 1. Initial setup

Locates the repository root by walking up from the current working directory until it finds the `src/` package, then adds it to `sys.path` so the `src.*` modules can be imported regardless of where Jupyter was launched from.

In [54]:
from pathlib import Path
import sys

# Find the root by searching the "src" folder
current = Path.cwd()

while current != current.parent:
    if (current / "src").exists():
        REPO_ROOT = current
        break
    current = current.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

## 2. Imports and LLM client configuration

Imports pandas, json, the pgmpy estimators and inference engine, and the FairMind causal modules (`build_sfm`, `fit_discrete_bayesian_model`, the effect functions). The llama.cpp endpoint is read from the `LLAMA_HOST`/`LLAMA_PORT` environment variables, falling back to `localhost:8080`, so the same notebook runs unchanged both locally and on Thor, where the sbatch script exports the address of the GPU node serving the model.

In [ ]:
import json
import os
import pandas as pd

from pgmpy.estimators import BayesianEstimator
from pgmpy.inference import VariableElimination
from src.graph import build_sfm
from src.model import fit_discrete_bayesian_model
from src.effects import (
    total_variation, total_effect,
    natural_direct_effect, natural_indirect_effect,
)
from src.llm import LLM_CONFIGS

LLAMA_HOST = os.environ.get("LLAMA_HOST", "localhost")
LLAMA_PORT = os.environ.get("LLAMA_PORT", "8080")
LLM_CONFIGS[0]["base_url"] = f"http://{LLAMA_HOST}:{LLAMA_PORT}/v1"
print(f"LLM endpoint configurato: http://{LLAMA_HOST}:{LLAMA_PORT}/v1")

## 3. Benchmark configuration

Defines the `CONFIG` dictionary for the Adult dataset: protected attribute (`S2_gender`, Female/Male), target (`T_income`, `>50K`), mediator (`hours-per-week`), confounder (`education`).

In [56]:
CONFIG = {
    "dataset_name": "adult",
    "csv_path": "../../data/processed/adult.csv",
    "target_col":  "T_income",
    "target_val":  ">50K",
    "protected":   "S2_gender",
    "x0": "Female",
    "x1": "Male",
    "mediators":   ["hours-per-week"],
    "confounders": ["education"],
}

## 4. FairMind - exact ground truth

Builds the Standard Fairness Model (SFM), fits the Bayesian Network with Laplace smoothing (pseudo-count alpha = 1) and computes TV, TE, DE and IE by formal causal inference. SE is derived as `TV - TE` (Eq. 3, Plečko and Bareinboim 2024). `education` (16 original levels) is grouped into 5 education tiers, as `hours-per-week` already is into bins: with 16×5=80 (z,w) combinations the LLM could not complete DE and IE, with 5×5=25 it can. It also returns the fitted BN: this is reused in `build_llm_prompt()` to compute the tables given to the LLM, so that FairMind and the LLM start from exactly the same numbers.

In [ ]:
import time

def run_fairmind(config: dict) -> tuple[dict, "DiscreteBayesianNetwork", int, float]:
    df = pd.read_csv(config["csv_path"])
    cols = (
        [config["protected"]]
        + config["mediators"]
        + config["confounders"]
        + [config["target_col"]]
    )
    df = df[cols].dropna()

    # Binned here, in-place, once: the BN fitted on this binned data is the
    # SAME instance build_llm_prompt() queries to build the tables given to
    # the LLM (supervisor's Point 4), so both sides start from identical
    # numbers on every cell, including the sparsest ones.
    if "hours-per-week" in df.columns:
        df["hours-per-week"] = pd.cut(
            df["hours-per-week"],
            bins=[0, 20, 35, 45, 60, 100],
            labels=["<=20", "21-35", "36-45", "46-60", ">60"],
            include_lowest=True,
        )

    # education: 16 levels -> 5 tiers. With 16 levels * 5 hour bins = 80
    # (z,w) combinations Qwen2.5-14B could not finish DE/IE: the response was
    # truncated even at max_tokens=16384, and a compact output format made it
    # worse (it returned invented values, e.g. TE identical to TV). With 5
    # tiers the combinations drop to 25. The tiers follow the standard
    # grouping used in the Adult literature (cf. Example 6 of the paper).
    if "education" in df.columns:
        education_tiers = {
            "Preschool": "<HS", "1st-4th": "<HS", "5th-6th": "<HS", "7th-8th": "<HS",
            "9th": "<HS", "10th": "<HS", "11th": "<HS", "12th": "<HS",
            "HS-grad": "HS-grad",
            "Some-college": "Some-college", "Assoc-acdm": "Some-college", "Assoc-voc": "Some-college",
            "Bachelors": "Bachelors",
            "Masters": "Grad", "Prof-school": "Grad", "Doctorate": "Grad",
        }
        df["education"] = df["education"].map(education_tiers)

    sfm = build_sfm(
        sensitive_attr=config["protected"],
        outcome_attr=config["target_col"],
        confounder_attrs=config["confounders"],
        mediator_attrs=config["mediators"],
        sorted_mediators=len(config["mediators"]) > 1,
        sorted_confounders=len(config["confounders"]) > 1,
    )
    # Laplace smoothing with pseudo-count alpha = 1 on every state of every
    # variable, i.e. P(state | parents) = (count + 1) / (N_parents + n_states),
    # matching the parameter estimation described in the reference paper.
    # pgmpy's "K2" is a shorthand for exactly this; the explicit dirichlet form
    # is used here because it states alpha = 1 in the code.
    bn = fit_discrete_bayesian_model(
        sfm=sfm,
        data=df,
        estimator_instance=(
            BayesianEstimator,
            {"prior_type": "dirichlet", "pseudo_counts": 1},
        ),
    )

    target = (config["target_col"], config["target_val"])
    x0, x1 = config["x0"], config["x1"]

    start = time.perf_counter()
    tv = total_variation(bn, target, config["protected"], x0, x1)
    te = total_effect(bn, target, config["protected"], x0, x1)
    effects = {
        "TV": tv,
        "TE": te,
        # SE = TV - TE (Eq. 3, Plecko & Bareinboim 2024), the same identity
        # the prompt asks the LLM to apply.
        "SE": tv - te,
        "DE": natural_direct_effect(bn, target, config["protected"], x0, x1),
        "IE": natural_indirect_effect(bn, target, config["protected"], x1, x0),
    }
    elapsed = time.perf_counter() - start

    return effects, bn, len(df), elapsed

ground_truth, bn, n_rows, fairmind_time = run_fairmind(CONFIG)
print(f"FairMind - elapsed time: {fairmind_time:.4f}s")
for k, v in ground_truth.items():
    print(f"  {k}: {v:.6f}")

## 5. Building the prompt for the LLM

Queries the already fitted Bayesian Network (via pgmpy's `VariableElimination`) to build the five conditional probability tables (P(Y|X), P(Z), P(Y|X,Z), P(W|X,Z), P(Y|X,W,Z)) - no longer empirical pandas frequencies over the raw dataset. Assembles the textual prompt with the identification formulae. It asks only for TV, TE, DE and IE (not SE, which is redundant: it follows by subtraction from TV and TE).

In [ ]:
from itertools import product

def _bn_states(bn, var: str) -> list:
    return bn.get_cpds(var).state_names[var]


def _bn_combos(bn, variables: list[str]) -> list[dict]:
    """All joint state combinations for a list of BN variables, as a list of
    dicts {variable: state}. Used to enumerate the table rows (one per
    combination) without hardcoding the states."""
    if not variables:
        return [{}]
    state_lists = [_bn_states(bn, v) for v in variables]
    return [dict(zip(variables, combo)) for combo in product(*state_lists)]


def build_llm_prompt(config: dict, bn, n_rows: int) -> str:
    """Builds the LLM prompt by querying DIRECTLY the Bayesian Network already
    fitted in run_fairmind() (same instance, same Laplace smoothing), instead of
    recomputing the probabilities with pandas on the raw dataset. Both sides
    then start from identical numbers on every table cell, including the
    sparsest ones (supervisor's Point 4).
    """
    protected = config["protected"]
    target_var = config["target_col"]
    target_val = config["target_val"]
    confounders = config["confounders"]
    mediators = config["mediators"]
    x0, x1 = config["x0"], config["x1"]

    ve = VariableElimination(bn)

    # --- 1. P(Y=y | X) ---
    rows = []
    for x in [x0, x1]:
        f = ve.query(variables=[target_var], evidence={protected: x}, show_progress=False)
        p = float(f.get_value(**{target_var: target_val}))
        rows.append({protected: x, "P(Y=y|X)": round(p, 4)})
    p_y_given_x = pd.DataFrame(rows)

    # --- 2. P(Z) - marginal distribution of the confounders ---
    z_factor = ve.query(variables=confounders, joint=True, show_progress=False)
    rows = []
    for z_combo in _bn_combos(bn, confounders):
        p = float(z_factor.get_value(**z_combo))
        rows.append({**z_combo, "P(Z)": round(p, 4)})
    p_z = pd.DataFrame(rows)

    # --- 3. P(Y=y | X, Z) ---
    rows = []
    for x in [x0, x1]:
        for z_combo in _bn_combos(bn, confounders):
            f = ve.query(variables=[target_var], evidence={protected: x, **z_combo}, show_progress=False)
            p = float(f.get_value(**{target_var: target_val}))
            rows.append({protected: x, **z_combo, "P(Y=y|X,Z)": round(p, 4)})
    p_y_given_xz = pd.DataFrame(rows)

    # --- 4. P(W | X, Z) ---
    rows = []
    for x in [x0, x1]:
        for z_combo in _bn_combos(bn, confounders):
            f = ve.query(variables=mediators, evidence={protected: x, **z_combo}, joint=True, show_progress=False)
            for w_combo in _bn_combos(bn, mediators):
                p = float(f.get_value(**w_combo))
                rows.append({protected: x, **z_combo, **w_combo, "P(W|X,Z)": round(p, 4)})
    p_w_given_xz = pd.DataFrame(rows)

    # --- 5. P(Y=y | X, W, Z) ---
    rows = []
    for x in [x0, x1]:
        for z_combo in _bn_combos(bn, confounders):
            for w_combo in _bn_combos(bn, mediators):
                f = ve.query(variables=[target_var], evidence={protected: x, **z_combo, **w_combo}, show_progress=False)
                p = float(f.get_value(**{target_var: target_val}))
                rows.append({protected: x, **z_combo, **w_combo, "P(Y=y|X,W,Z)": round(p, 4)})
    p_y_given_xwz = pd.DataFrame(rows)

    def to_compact_csv(d: pd.DataFrame) -> str:
        return d.to_csv(index=False)

    return f"""You are a causal fairness expert. Compute four causal fairness effects
using the Standard Fairness Model (SFM) by Plecko and Bareinboim (2024).

You are given PRE-AGGREGATED CONDITIONAL PROBABILITY TABLES computed from a fitted
Bayesian Network (n={n_rows} training rows, Laplace-smoothed CPDs, alpha=1). Use these tables
directly — do not assume access to raw data.
Note: "hours-per-week" has been discretized into bins: <=20, 21-35, 36-45, 46-60, >60.
Note: "education" has been grouped into tiers: <HS, HS-grad, Some-college, Bachelors, Grad.

VARIABLE ROLES:
- X (protected): "{protected}", x0="{x0}", x1="{x1}"
- Y (target):    "{target_var}", target state="{target_val}"
- W (mediators): {mediators}
- Z (confounders): {confounders}

TABLE 1 — P(Y=y | X):
{to_compact_csv(p_y_given_x)}

TABLE 2 — P(Z):
{to_compact_csv(p_z)}

TABLE 3 — P(Y=y | X, Z):
{to_compact_csv(p_y_given_xz)}

TABLE 4 — P(W | X, Z):
{to_compact_csv(p_w_given_xz)}

TABLE 5 — P(Y=y | X, W, Z):
{to_compact_csv(p_y_given_xwz)}

IDENTIFICATION FORMULAE (use these exactly, aggregating over TABLE rows as needed):
- TV = P(Y=y | X=x1) - P(Y=y | X=x0)                    [from TABLE 1]
- TE = sum_z [P(Y=y|x1,z) - P(Y=y|x0,z)] * P(z)          [from TABLE 3, TABLE 2]
- DE = sum_z,w [P(Y=y|x1,w,z) - P(Y=y|x0,w,z)] * P(w|x0,z) * P(z)   [from TABLE 5, TABLE 4, TABLE 2]
- IE = sum_z,w P(Y=y|x1,w,z) * [P(w|x0,z) - P(w|x1,z)] * P(z)       [from TABLE 5, TABLE 4, TABLE 2]

Note: IE is requested in its REVERSE form (x1 -> x0), which is the one entering
the decomposition TE = DE - IE (Plecko & Bareinboim, 2024, Prop. 2). Keep x1 in the
outcome term and subtract in the order written above; do not rewrite it the other way.

Note: the Spurious Effect (SE) is NOT requested here — it is fully determined
by SE = TV - TE (Plecko & Bareinboim, 2024), so it is derived afterwards
from your TV and TE values rather than computed independently.

INSTRUCTIONS:
For DE and IE, the sums run over EVERY combination of z (each row of TABLE 2) and
w (each bin of hours-per-week). For each (z,w) pair you MUST look up the matching
row in TABLE 4 and TABLE 5 by z and w together — do not skip or approximate any term.

Show your work as a short step-by-step calculation for DE and IE (one line per (z,w)
term is fine, or grouped by z), THEN give the final answer.

End your response with a line "FINAL_JSON:" followed by ONLY the JSON object below,
with no markdown formatting:
{{
  "TV": <float>,
  "TE": <float>,
  "DE": <float>,
  "IE": <float>
}}"""

prompt = build_llm_prompt(CONFIG, bn, n_rows)
print(prompt[:2000], "\n[...]")
print(f"\nTotal prompt length (chars): {len(prompt)}")

## 6. Calling the LLM (Qwen2.5-14B) and parsing its response

Sends the prompt with the pre-aggregated tables to Qwen2.5-14B via llama.cpp, collects timing and token metrics, extracts the JSON (TV, TE, DE, IE) from the response with a regex and parses it. SE is then computed as `TV - TE` on the values returned by the LLM.

In [ ]:
from src.llm import call_llm

# max_tokens=16384: with education binned into 5 tiers (25 (z,w) combinations
# instead of 80) the original show-your-work prompt should need far less, but
# we keep a wide budget as a safety net.
# cache_prompt=False: llama.cpp routes a request to the slot with the
# longest common prefix and reuses its KV cache, which made identical
# requests diverge even at temperature 0 (see call_llm's docstring).
# Disabled here so the run is reproducible.
llm_effects, llm_usage, llm_time = call_llm(
    prompt, max_tokens=16384, cache_prompt=False
)

# SE is not asked of the LLM (see the note in the prompt): it is derived here
# with the same identity used for the ground truth (SE = TV - TE), so the SE
# comparison reflects only the LLM's TV/TE errors. Reordered explicitly to
# TV,TE,SE,DE,IE because call_llm() appends SE at the end of the dict.
llm_effects["SE"] = llm_effects["TV"] - llm_effects["TE"]
llm_effects = {k: llm_effects[k] for k in ["TV", "TE", "SE", "DE", "IE"]}

print(f"LLM - time: {llm_time:.4f}s")
print(f"Token: input={llm_usage['input_tokens']}, "
      f"output={llm_usage['output_tokens']}, "
      f"total={llm_usage['total_tokens']}")
print(json.dumps(llm_effects, indent=2))

## 7. Comparing FairMind vs LLM - the discrepancies table

Computes the absolute and relative percentage error between the ground truth (FairMind) and the LLM prediction for each of the 5 causal effects, producing a summary table.

In [60]:
def compute_discrepancies(ground_truth: dict, llm_effects: dict) -> pd.DataFrame:
    rows = []
    for effect in ["TV", "TE", "SE", "DE", "IE"]:
        gt  = ground_truth.get(effect, float("nan"))
        llm_val = float(llm_effects.get(effect, float("nan")))
        abs_err = abs(gt - llm_val)
        rel_err = abs_err / abs(gt) if abs(gt) > 1e-9 else float("nan")
        rows.append({
            "effect":      effect,
            "fairmind":    round(gt,  6),
            "llm":         round(llm_val, 6),
            "abs_error":   round(abs_err, 6),
            "rel_error_%": round(rel_err * 100, 2) if not pd.isna(rel_err) else float("nan"),
        })
    return pd.DataFrame(rows)

discrepancies = compute_discrepancies(ground_truth, llm_effects)
print(discrepancies.to_string(index=False))

effect  fairmind       gpt  abs_error  rel_error_%
    TV  0.194470  0.194500   0.000030         0.02
    TE  0.183161  0.231167   0.048005        26.21
    SE -0.007296 -0.036667   0.029371       402.56
    DE  0.137049  0.083167   0.053883        39.32
    IE -0.046112 -0.047000   0.000888         1.93


## 8. Saving results to disk

Saves the whole benchmark result to a JSON file inside `benchmark_results/`, including configuration, FairMind effects, LLM effects, discrepancies, token metrics and timing.

In [61]:
def save_results(config, ground_truth, llm_effects, discrepancies, usage, fairmind_time, llm_time):
    import os, datetime
    os.makedirs("benchmark_results", exist_ok=True)
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    fname = f"benchmark_results/{config['dataset_name']}_{ts}.json"

    # The model's working is bulky, so it goes at the top level of the JSON
    # instead of inside token_usage, which stays a block of plain counters.
    usage = dict(usage)
    raw_response = usage.pop("raw_response", None)

    out = {
        "dataset":       config["dataset_name"],
        "config":        {k: v for k, v in config.items() if k != "csv_path"},
        "fairmind":      ground_truth,
        "llm":           llm_effects,
        "discrepancies": discrepancies.to_dict(orient="records"),
        "token_usage":   usage,
        "llm_raw_response": raw_response,
        "timing": {
            "fairmind_seconds": round(fairmind_time, 4),
            "llm_seconds":      round(llm_time, 4),
        },
    }
    with open(fname, "w") as f:
        json.dump(out, f, indent=2)
    print(f"Saved: {fname}")

save_results(CONFIG, ground_truth, llm_effects, discrepancies, llm_usage, fairmind_time, llm_time)

Saved: benchmark_results/adult_20260712_161949.json
